# Agentic AI + XAI - run on Google Colab

Runs the WAF agent with **meta-llama/Llama-3.1-8B-Instruct**. The model is downloaded from Hugging Face when `main.py` starts (about 16 GB, cached for the rest of the session).

**Before you start**
1. *Runtime > Change runtime type > T4 GPU* (or L4/A100).
2. Accept Meta's license at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
3. Create a token at https://huggingface.co/settings/tokens and add it in Colab's **Secrets** panel (key icon) as `HF_TOKEN`, with notebook access on.
4. Upload this project folder to Google Drive (including the `Web-Application-Attack-Datasets-master` folder).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change this to where the project folder sits in your Drive
%cd "/content/drive/MyDrive/Agentic AI XAI"

In [ ]:
!pip install -q -r requirements-colab.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # passed on to main.py
# Optional overrides:
# os.environ['LLAMA_QUANTIZATION'] = '4bit'   # auto | 4bit | 8bit | none
# os.environ['MAX_NEW_TOKENS'] = '512'

Check the MCP servers and tools (no model needed):

In [ ]:
!python main.py --check-tools

Run the agent. The first run downloads Llama-3.1-8B-Instruct, then loads it on the GPU (4-bit on a T4).

In [ ]:
!python main.py --n 4

In [ ]:
!python main.py --dataset anomalous_test --index 0

Inspect the traces (input for the XAI phase):

In [ ]:
import glob, json, pandas as pd
rows = [json.loads(l) for f in sorted(glob.glob('runs/*.jsonl')) for l in open(f, encoding='utf-8')]
pd.DataFrame([{k: r[k] for k in ['request_id','ground_truth','predicted','correct','tool_sequence']} for r in rows])

## XAI: LIME - why did the agent decide that?

LIME removes random tokens (or whole parameters) from the request a few hundred times, asks Llama for the decision probability each time (one forward pass with the agent's exact prompt and tools), and fits a weighted linear model. Its weights show which parts of the request pushed the decision.

Targets: `verdict` (normal/attack), `action` (allow/block), `first_tool` (which tool it calls first). 300 samples takes a few minutes on a T4.

**Option A - command line** (reloads the model each time):

In [ ]:
!python explain.py --dataset anomalous_test --index 1 --target verdict --granularity token --num-samples 300

In [ ]:
# Explain the requests from the agent run above; also reports how often the probe agrees with the agent
import glob
latest = sorted(glob.glob('runs/run_*.jsonl'))[-1]
!python explain.py --trace {latest} --target action --granularity param --num-samples 150

**Option B - in this notebook** (model stays loaded, faster when you explain many requests):

In [ ]:
from agent_xai.agent import build_llm
from agent_xai.data_loader import get_request
from explain import tool_schemas, explain_many

llm = build_llm('meta-llama/Llama-3.1-8B-Instruct')   # loads from the HF cache after the first download
schemas = await tool_schemas()

In [ ]:
from IPython.display import HTML, display
from agent_xai.xai.report import to_html

items = [(get_request('anomalous_test', 1), None), (get_request('normal_test', 5), None)]
results = explain_many(llm, schemas, items, target='verdict', granularity='token', num_samples=300)
for (req, _), e in zip(items, results):
    display(HTML(to_html(e, req)))